In [1]:
import requests
from bs4 import BeautifulSoup
import sqlite3
from datetime import datetime
import time
import re
import pandas as pd



In [ ]:
import requests
from bs4 import BeautifulSoup
from dateutil import parser
import sqlite3

# Connect or create the SQLite DB
conn = sqlite3.connect("timeline.db")
cursor = conn.cursor()

# Create the table if it doesn't exist
cursor.execute("""
    CREATE TABLE IF NOT EXISTS timeline (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        event_date DATE,
        title TEXT,
        description TEXT
    )
""")

def scrape_historycooperative():
    url = "https://historycooperative.org/ww2-dates/"
    source = "historycooperative.org"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    }

    print(f"Scraping {source}...")

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, "html.parser")
        entry_div = soup.find("div", class_="entry-content")

        if not entry_div:
            print("Could not find timeline content")
            return

        entries = entry_div.find_all("p")
        saved_count = 0

        for idx, p in enumerate(entries, start=1):
            strong_tag = p.find("strong")
            if strong_tag:
                date_text = strong_tag.get_text(strip=True)
                # Clean and parse the date
                try:
                    clean_date = date_text.split('–')[0].strip()
                    date_obj = parser.parse(clean_date, fuzzy=True).date()

                    #Attempt to add clean text to write the description
                    clean_title = date_text.split('–')[1].strip()

                except Exception as e:
                    print(f"Skipping invalid date: {date_text} | Error: {e}")
                    continue

                # Remove strong tag to get the rest of the description
                strong_tag.decompose()
                desc_text = p.get_text(strip=True)

                print(f"{idx}. {date_obj} - {clean_title}")
                cursor.execute("INSERT INTO timeline (event_date, title, description) VALUES (?, ?, ?)",
                               (date_obj, clean_title, desc_text))
                saved_count += 1

        conn.commit()
        print(f"Saved {saved_count} entries from {source}.")

    except Exception as e:
        print(f"Error scraping {source}: {e}")

    finally:
        conn.close()

scrape_historycooperative()


In [ ]:
scrape_historycooperative()

In [ ]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import sqlite3
import time

def create_database():
    """Create and initialize the SQLite database"""
    conn = sqlite3.connect('ww2_timeline.db')
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS events (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            date DATE NOT NULL,
            year TEXT NOT NULL,
            description TEXT NOT NULL,
            source TEXT NOT NULL,
            UNIQUE(date, description)
        )
    ''')
    conn.commit()
    return conn, cursor

def scrape_historycooperative(conn, cursor):
    """Scrape WWII timeline from historycooperative.org"""
    url = "https://historycooperative.org/ww2-dates/"
    source = "historycooperative.org"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept-Language": "en-US,en;q=0.9"
    }
    
    print(f"\nScraping {source} timeline...")
    
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Find the main content container - updated selector
        content = soup.find('div', class_='entry-content') or soup.find('article')
        
        if not content:
            print("Error: Could not find content container")
            print("Here's the page structure for debugging:")
            print(soup.prettify()[:2000])  # Print first 2000 chars
            return 0
        
        saved_count = 0
        current_year = None
        
        # Process all elements in the content
        for element in content.find_all(recursive=False):
            # Extract year from h2 headings
            if element.name == 'h2' and 'wp-block-heading' in element.get('class', []):
                current_year = element.get_text(strip=True)
                print(f"\nProcessing year: {current_year}")
            
            # Extract dates and events
            elif element.name == 'p':
                # Check for date paragraphs (they contain the date in bold)
                if element.find('strong'):
                    date_text = element.get_text(strip=True)
                    
                    try:
                        # Clean date text and parse
                        date_text = date_text.replace('\xa0', ' ')  # Replace non-breaking spaces
                        date_obj = datetime.strptime(date_text, "%B %d, %Y").date()
                        
                        # Get the next paragraph which contains the description
                        desc_element = element.find_next_sibling('p')
                        if desc_element and not desc_element.find('strong'):
                            description = desc_element.get_text(strip=True)
                            
                            # Save to database
                            cursor.execute(
                                '''INSERT INTO events 
                                (date, year, description, source) 
                                VALUES (?, ?, ?, ?)''',
                                (date_obj, current_year, description, source)
                            )
                            conn.commit()
                            saved_count += 1
                            print(f"✓ {date_obj}: {description[:60]}...")
                                                                
                    except ValueError as e:
                        print(f"⚠️ Could not parse date: {date_text} - {str(e)}")
                        continue
        
        print(f"\nSuccessfully saved {saved_count} new events from {source}")
        return saved_count
        
    except requests.exceptions.RequestException as e:
        print(f"Network error: {str(e)}")
        return 0
    except Exception as e:
        print(f"Unexpected error: {str(e)}")
        return 0

def main():
    try:
        # Initialize database
        conn, cursor = create_database()
        
        # Start scraping
        start_time = time.time()
        events_saved = scrape_historycooperative(conn, cursor)
        
        if events_saved == 0:
            print("\nWarning: No events were saved. Possible reasons:")
            print("- The website structure has changed")
            print("- Your IP might be blocked (try using a VPN)")
            print("- Check the debug output above for clues")
        
        print(f"\nCompleted in {time.time() - start_time:.2f} seconds")
        
    finally:
        # Ensure connection is closed
        if 'conn' in locals():
            conn.close()
            print("Database connection closed")

if __name__ == "__main__":
    main()